In [13]:
import pandas as pd

print("1. Chargement et vérification des données...")

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

print("\n--- Aperçu des données ---")
print(df.head())
print("\n--- Vérification des valeurs manquantes ---")
print(df.isnull().sum())

1. Chargement et vérification des données...

--- Aperçu des données ---
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

-

In [14]:
print("Nettoyage des données...")

# 1. Traiter les valeurs manquantes (total_bedrooms) : remplacement par la médiane
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# 2. Encoder la variable catégorielle 'ocean_proximity' en variables numériques
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

# 3. Renommer la cible pour rester cohérent avec la suite du notebook
df = df.rename(columns={'median_house_value': 'Prix'})

print("\n--- Vérification après nettoyage ---")
print(df.isnull().sum())
print("\n--- Statistiques de base du Dataset ---")
print(df.describe().loc[['mean', 'min', 'max']])
print("\n--- Aperçu des colonnes finales ---")
print(df.columns.tolist())

Nettoyage des données...

--- Vérification après nettoyage ---
longitude                     0
latitude                      0
housing_median_age            0
total_rooms                   0
total_bedrooms                0
population                    0
households                    0
median_income                 0
Prix                          0
ocean_proximity_INLAND        0
ocean_proximity_ISLAND        0
ocean_proximity_NEAR BAY      0
ocean_proximity_NEAR OCEAN    0
dtype: int64

--- Statistiques de base du Dataset ---
       longitude   latitude  housing_median_age   total_rooms  total_bedrooms  \
mean -119.569704  35.631861           28.639486   2635.763081      536.838857   
min  -124.350000  32.540000            1.000000      2.000000        1.000000   
max  -114.310000  41.950000           52.000000  39320.000000     6445.000000   

        population  households  median_income           Prix  
mean   1425.476744   499.53968       3.870671  206855.816909  
min       3.0000

In [15]:
from sklearn.model_selection import train_test_split

print("2. Préparation des données d'apprentissage...")
X = df.drop('Prix', axis=1)
y = df['Prix']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

2. Préparation des données d'apprentissage...


In [16]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

print("3. Entraînement des intelligences artificielles...")

# Modèle 1 : Régression Linéaire
lr_modele = LinearRegression()
lr_modele.fit(X_train, y_train)
lr_predictions = lr_modele.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))

# Modèle 2 : Random Forest (optimisé pour rester léger en taille)
rf_modele = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_modele.fit(X_train, y_train)
rf_predictions = rf_modele.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

3. Entraînement des intelligences artificielles...


In [17]:
print("\n4. Résultats du Benchmark des modèles :")
print(f"   📉 Erreur Régression Linéaire : {lr_rmse:.2f} $")
print(f"   🏆 Erreur Random Forest       : {rf_rmse:.2f} $")
print("   -> Le Random Forest gagne largement, c'est lui qu'on mettra en production !")


4. Résultats du Benchmark des modèles :
   📉 Erreur Régression Linéaire : 70060.52 $
   🏆 Erreur Random Forest       : 49998.30 $
   -> Le Random Forest gagne largement, c'est lui qu'on mettra en production !


In [18]:
import joblib
import os

print("5. Sauvegarde des fichiers MLOps...")

joblib.dump(rf_modele, 'modele_immobilier_rf.joblib', compress=3)
joblib.dump(list(X.columns), 'colonnes_features.joblib')

taille_mb = os.path.getsize('modele_immobilier_rf.joblib') / (1024*1024)
print(f"✅ Terminé ! Taille du fichier modèle : {taille_mb:.2f} MB")

5. Sauvegarde des fichiers MLOps...
✅ Terminé ! Taille du fichier modèle : 5.73 MB
